In [0]:
spark.sql("USE CATALOG retail_lakehouse")
spark.sql("USE SCHEMA bronze")

In [0]:
orders_path = "/Volumes/retail_lakehouse/bronze/source_data/order_system/orders.csv"

order_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(orders_path)

)

display(order_df)

In [0]:
order_df.printSchema()

In [0]:
display(order_df.limit(10))

In [0]:
# load the data into bronze layer 

order_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_lakehouse.bronze.orders")

    

In [0]:
%sql
SELECT * FROM retail_lakehouse.bronze.orders 

In [0]:
# load all rest of files into bronze layer

order_items_path = "/Volumes/retail_lakehouse/bronze/source_data/order_system/order_items.csv"
order_items_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(order_items_path)
)

display(order_items_df)

(
order_items_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_lakehouse.bronze.order_items") 
)

In [0]:
payments_df = (
    spark.read
    .option("multiline", True)
    .option("inferSchema", True)
    .json("/Volumes/retail_lakehouse/bronze/source_data/payment_system/payments.json")
)

display(payments_df)

(
    payments_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("retail_lakehouse.bronze.payments")
)

In [0]:
customer_df = (
    spark.read
    .option("multiline", True)
    .option("inferSchema", True)
    .json("/Volumes/retail_lakehouse/bronze/source_data/crm/customers.json")
)

display(customer_df)

(
    customer_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("retail_lakehouse.bronze.customers")
)


In [0]:
# read the data product system

product_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/retail_lakehouse/bronze/source_data/product_system/products.csv")

)

display(product_df)

(
    product_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("retail_lakehouse.bronze.products")
)

In [0]:
# read the shipping system files

ship_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/retail_lakehouse/bronze/source_data/shipping_system/shipments.csv")
)

display(ship_df)

(
    ship_df.write \
        .format("delta") \
            .mode("overwrite") \
                .saveAsTable("retail_lakehouse.bronze.shipping")
)

In [0]:
%sql

--  get the counts from all the tables bronze layer

SELECT 'customers' AS table_name, COUNT(*) As row_count
    FROM retail_lakehouse.bronze.customers
UNION ALL

SELECT 'order_items' AS table_name, count(*) AS row_count
    FROM retail_lakehouse.bronze.order_items
UNION ALL

SELECT 'order' AS table_name, count(*) As row_count
    FROM retail_lakehouse.bronze.orders

UNION ALL

SELECT 'payments' AS table_name, count(*) AS row_count
    FROM retail_lakehouse.bronze.payments

UNION ALL

SELECT 'products' AS table_name, count(*) AS row_count
    FROM retail_lakehouse.bronze.products

UNION ALL

SELECT 'ship' AS table_name, count(*) AS row_count
    FROM retail_lakehouse.bronze.shipping